# WAV 일회성 통합기

여러 개의 WAV 파일을 **파일명 순서대로 하나의 WAV로 이어붙이는** 노트북입니다.

- FFmpeg 사용 안 함
- Conda 패키지 설치 불필요
- Python 표준 라이브러리 `wave`만 사용
- 원본 WAV의 샘플레이트 / 채널 / 비트 깊이가 모두 같을 때 가장 안전하게 동작
- 통합 후 각 원본 파일이 어느 시간대에 들어갔는지 `CSV manifest`도 생성

> 사용법: 아래 `INPUT_DIR`만 WAV 파일들이 들어 있는 폴더로 바꾼 뒤 위에서부터 순서대로 실행하세요.

In [ ]:
from pathlib import Path

# =========================
# 사용자 설정
# =========================

# WAV 파일들이 들어 있는 폴더
INPUT_DIR = Path(r"C:\Users\hokusho\Desktop\wav_merge")

# 생성 파일명
OUTPUT_WAV = INPUT_DIR / "merged.wav"
MANIFEST_CSV = INPUT_DIR / "merged_manifest.csv"

# 하위 폴더까지 찾을지 여부
RECURSIVE = False

# 파일 사이에 넣을 무음 길이(초)
# 바로 이어붙이려면 0.0
GAP_SECONDS = 0.0

print("입력 폴더 :", INPUT_DIR)
print("출력 WAV :", OUTPUT_WAV)
print("목록 CSV :", MANIFEST_CSV)

In [ ]:
import re

def natural_key(path: Path):
    """1.wav, 2.wav, 10.wav 순서가 되도록 자연 정렬."""
    return [
        int(part) if part.isdigit() else part.lower()
        for part in re.split(r"(\d+)", path.name)
    ]

pattern = "**/*.wav" if RECURSIVE else "*.wav"

wav_files = [
    p for p in INPUT_DIR.glob(pattern)
    if p.is_file() and p.resolve() != OUTPUT_WAV.resolve()
]

wav_files = sorted(wav_files, key=natural_key)

print(f"발견된 WAV 파일: {len(wav_files)}개")
for i, p in enumerate(wav_files, 1):
    print(f"{i:>3}. {p.name}")

if not wav_files:
    raise FileNotFoundError("입력 폴더에서 WAV 파일을 찾지 못했습니다.")

In [ ]:
import wave
from dataclasses import dataclass

@dataclass
class WavInfo:
    path: Path
    channels: int
    sample_width: int
    frame_rate: int
    frames: int
    comptype: str
    duration: float

def read_wav_info(path: Path) -> WavInfo:
    with wave.open(str(path), "rb") as w:
        channels = w.getnchannels()
        sample_width = w.getsampwidth()
        frame_rate = w.getframerate()
        frames = w.getnframes()
        comptype = w.getcomptype()

    return WavInfo(
        path=path,
        channels=channels,
        sample_width=sample_width,
        frame_rate=frame_rate,
        frames=frames,
        comptype=comptype,
        duration=frames / frame_rate if frame_rate else 0.0,
    )

infos = [read_wav_info(p) for p in wav_files]

print("WAV 형식 확인")
print("-" * 80)

for info in infos:
    print(
        f"{info.path.name} | "
        f"{info.frame_rate} Hz | "
        f"{info.channels} ch | "
        f"{info.sample_width * 8} bit | "
        f"{info.duration:.2f} sec"
    )

In [ ]:
# 모든 WAV 파일의 포맷이 같은지 검사
base = infos[0]

base_format = (
    base.channels,
    base.sample_width,
    base.frame_rate,
    base.comptype,
)

mismatches = []

for info in infos[1:]:
    current_format = (
        info.channels,
        info.sample_width,
        info.frame_rate,
        info.comptype,
    )
    if current_format != base_format:
        mismatches.append(info)

if mismatches:
    print("❌ WAV 포맷이 서로 달라서 안전한 단순 병합을 중단합니다.\n")
    print(
        f"기준: {base.path.name} | "
        f"{base.frame_rate} Hz | {base.channels} ch | {base.sample_width*8} bit"
    )
    print("\n다른 파일:")
    for info in mismatches:
        print(
            f"- {info.path.name} | "
            f"{info.frame_rate} Hz | {info.channels} ch | {info.sample_width*8} bit"
        )
    raise RuntimeError(
        "샘플레이트/채널/비트 깊이가 다른 WAV가 있습니다. "
        "이 경우 변환 후 병합해야 합니다."
    )

print("✅ 모든 WAV 파일의 포맷이 동일합니다.")
print(
    f"{base.frame_rate} Hz / "
    f"{base.channels} ch / "
    f"{base.sample_width * 8} bit"
)

In [ ]:
import csv

# 무음 frame 생성
gap_frames = int(round(GAP_SECONDS * base.frame_rate))
silence_frame = b"\x00" * base.sample_width * base.channels
silence_bytes = silence_frame * gap_frames

manifest_rows = []
current_time = 0.0

with wave.open(str(OUTPUT_WAV), "wb") as out:
    out.setnchannels(base.channels)
    out.setsampwidth(base.sample_width)
    out.setframerate(base.frame_rate)
    out.setcomptype(base.comptype, "not compressed")

    for idx, info in enumerate(infos, 1):
        start_sec = current_time

        with wave.open(str(info.path), "rb") as src:
            # 메모리 과사용 방지를 위해 chunk 단위로 복사
            while True:
                chunk = src.readframes(65536)
                if not chunk:
                    break
                out.writeframesraw(chunk)

        end_sec = start_sec + info.duration

        manifest_rows.append({
            "index": idx,
            "filename": info.path.name,
            "start_sec": round(start_sec, 3),
            "end_sec": round(end_sec, 3),
            "duration_sec": round(info.duration, 3),
        })

        current_time = end_sec

        # 마지막 파일 뒤에는 무음 추가하지 않음
        if idx < len(infos) and gap_frames > 0:
            out.writeframesraw(silence_bytes)
            current_time += GAP_SECONDS

with MANIFEST_CSV.open("w", newline="", encoding="utf-8-sig") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=[
            "index",
            "filename",
            "start_sec",
            "end_sec",
            "duration_sec",
        ],
    )
    writer.writeheader()
    writer.writerows(manifest_rows)

print("✅ 병합 완료")
print("WAV :", OUTPUT_WAV)
print("CSV :", MANIFEST_CSV)
print(f"총 길이: {current_time:.2f}초 ({current_time/60:.2f}분)")

In [ ]:
# 결과 간단 검증
with wave.open(str(OUTPUT_WAV), "rb") as w:
    merged_duration = w.getnframes() / w.getframerate()

print("통합 WAV 정보")
print(f"- Sample rate : {w.getframerate()} Hz")
print(f"- Channels    : {w.getnchannels()}")
print(f"- Bit depth   : {w.getsampwidth() * 8} bit")
print(f"- Duration    : {merged_duration:.2f} sec ({merged_duration/60:.2f} min)")

print("\n구간 목록")
for row in manifest_rows:
    print(
        f"{row['index']:>3}. "
        f"{row['start_sec']:>8.3f} ~ {row['end_sec']:>8.3f} sec | "
        f"{row['filename']}"
    )

## 결과

실행이 끝나면 입력 폴더에 아래 두 파일이 생성됩니다.

- `merged.wav` : 모든 WAV를 순서대로 이어붙인 결과
- `merged_manifest.csv` : 각 원본 WAV가 합쳐진 파일의 몇 초 구간에 위치하는지 기록

예:

```text
1.wav  -> 0.000 ~ 12.481 sec
2.wav  -> 12.481 ~ 26.922 sec
3.wav  -> 26.922 ~ 41.105 sec
```

이 CSV는 나중에 STT/화자분리 테스트 결과와 원본 데이터를 비교할 때도 사용할 수 있습니다.

### 주의
이 노트북은 DLL 충돌을 피하기 위해 FFmpeg를 사용하지 않습니다.  
따라서 샘플레이트, 채널 수, 비트 깊이가 다른 WAV가 섞여 있으면 자동으로 멈춥니다.